In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [7]:
#1. Data Understanding
df = pd.read_csv(r"C:\Users\aryan\Downloads\archive\IMDB Dataset.csv")
df.head()
print("Shape:", df.shape)
print(df['sentiment'].value_counts())
print(df['review'][0])
df.rename(columns={'review': 'text', 'sentiment': 'label'}, inplace=True)
df['label'] = df['label'].map({'positive': 1, 'negative': 0})

Shape: (50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreement

In [9]:
#2. NLP Preprocessing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)
df['clean_text'] = df['text'].apply(clean_text)
df[['text','clean_text']].head()

,text,clean_text
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [11]:
#3. Feature Engineering
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = tfidf.fit_transform(df['clean_text'])
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
#4. Model Building
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [15]:
#5. Model Evaluation
def evaluate(name, y_test, y_pred):
    print(name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("-"*50)
evaluate("Logistic Regression", y_test, y_pred_lr)
evaluate("Naive Bayes", y_test, y_pred_nb)
evaluate("Decision Tree", y_test, y_pred_dt)

Logistic Regression
Accuracy: 0.8957
Precision: 0.8848228043143297
Recall: 0.9116888271482437
F1 Score: 0.8980549310917799
--------------------------------------------------
Naive Bayes
Accuracy: 0.8666
Precision: 0.8592204770215242
Recall: 0.8793411391149037
F1 Score: 0.8691643781875246
--------------------------------------------------
Decision Tree
Accuracy: 0.7158
Precision: 0.7212487411883183
Recall: 0.710656876364358
F1 Score: 0.7159136345461815
--------------------------------------------------


In [ ]:
# 6. Comparison & Insights
a. Logistic Regression performed best (~90% accuracy)
b. Naive Bayes is faster but slightly less accurate
c. Decision Tree shows overfitting
# Observations:
a)TF-IDF improves performance
b) Lemmatization helps normalize text
c) Bigrams improve context understanding
# Conclusion:
Logistic Regression with TF-IDF is the best combination.